<a href="https://colab.research.google.com/github/71percentbanana/gridathon/blob/main/gridathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Gridathon**

## Load and clean Data

In [9]:
import pandas as pd

df=pd.read_csv('https://raw.githubusercontent.com/71percentbanana/gridathon/refs/heads/main/train.csv')

df['Weather'].value_counts(dropna=False)
df['RoadType'].value_counts(dropna=False)
df['LargeVehicles'].value_counts(dropna=False)
df['Landmarks'].value_counts(dropna=False)
df['geohash'].nunique()
df['timestamp'].sample(10)
df['timestamp'].nunique()
df.groupby('Weather')['demand'].mean()
df.groupby('RoadType')['demand'].mean()
df.groupby('NumberofLanes')['demand'].mean()
df.groupby('LargeVehicles')['demand'].mean()
df.groupby('Landmarks')['demand'].mean()
df.groupby('day')['demand'].mean()
df.groupby('timestamp')['demand'].mean().sort_values(ascending=False).head(10)
df.groupby('timestamp')['demand'].mean().sort_index().head(20)
df.groupby('timestamp')['demand'].mean().sort_index().tail(20)
df['timestamp'].unique()[:20]
df.groupby('RoadType')['NumberofLanes'].mean()
df.groupby('RoadType')['LargeVehicles'].value_counts()
df.groupby(['RoadType', 'LargeVehicles'])['demand'].mean()
df.groupby('RoadType')['Temperature'].mean()
df.groupby('Weather')['Temperature'].mean()
df[['Temperature', 'demand']].corr()
df['demand'].describe()

df.head()
df.isnull().sum()

df['Temperature'] = df['Temperature'].fillna(
    df['Temperature'].median()
)

df['RoadType'] = df['RoadType'].fillna('Unknown')

df['Weather'] = df['Weather'].fillna('Unknown')

df.isnull().sum()

df['hour'] = df['timestamp'].str.split(':').str[0].astype(int)

df['minute'] = df['timestamp'].str.split(':').str[1].astype(int)


df[['timestamp', 'hour', 'minute']].head(10)

X = df.drop('demand', axis=1)
y = df['demand']

print(X.shape)
print(y.shape)

(77299, 12)
(77299,)


# split code

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_val.shape)
print(y_train.shape)
print(y_val.shape)

(61839, 12)
(15460, 12)
(61839,)
(15460,)


# CatBoost

In [12]:
!pip install catboost

from catboost import CatBoostRegressor
cat_features = [
    'geohash',
    'timestamp',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather'
]

cat_features

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.0 MB/s eta 0:00:00


['geohash', 'timestamp', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']

# Train

In [13]:
model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=8,
    loss_function='RMSE',
    verbose=100
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_features
)

0:	learn: 0.1369069	total: 134ms	remaining: 1m 6s
100:	learn: 0.0434038	total: 15.6s	remaining: 1m 1s
200:	learn: 0.0396015	total: 25.7s	remaining: 38.2s
300:	learn: 0.0374298	total: 35.6s	remaining: 23.5s
400:	learn: 0.0360339	total: 46.3s	remaining: 11.4s
499:	learn: 0.0347576	total: 54.4s	remaining: 0us


CatBoostRegressor(depth=8, iterations=500, learning_rate=0.05, loss_function='RMSE', verbose=100)

# Test

In [14]:
from sklearn.metrics import r2_score

preds = model.predict(X_val)

r2 = r2_score(y_val, preds)

print("R2 Score:", r2)
print("Competition Score:", r2 * 100)

R2 Score: 0.9356137885853328
Competition Score: 93.56137885853329
